# Feature Preparation: Public Set + Survey Set

Prepares features from:
1. **Public set** — UCI Student Performance
2. **Survey set** — Your cleaned survey data (`survey_cleaned_python.xlsx`)

Both are processed consistently for modeling.

The committed survey feature table contains only derived fields. The original cleaned survey workbook is a group input. This notebook prepares an exploratory combined table using UCI Student Performance (dataset 320). Part C fits separate GPA classifiers to the group survey and UCI Higher Education (dataset 856).


In [1]:
import pandas as pd
import numpy as np

print('Libraries loaded')

Libraries loaded


## Part 1: Load Survey Data

In [ ]:
# Load your cleaned survey data
survey = pd.read_excel(
    'survey_cleaned_python.xlsx',
    sheet_name='Cleaned Survey'
)

print(f'Survey shape: {survey.shape}')
print('\nColumns:')
for col in survey.columns:
    print(f'  - {col}')
survey.head(3)

## Part 2: Prepare Survey Features

In [ ]:
# Column names from your survey
GPA_COL = 'the range your Gpa grade of last semester'
STUDY_COL = 'How much time you spent on studying per week ( outside of class time)'

# Map GPA ranges to numeric midpoint (for regression)
gpa_map = {
    'below 2.5': 2.25,
    '2.5-2.99': 2.75,
    '3.0-3.49': 3.25,
    '3.5 above': 3.75
}

# Map study time to ordinal (1-6)
study_map = {
    '0-5 hours': 1,
    '6-10 hours': 2,
    '11-15 hours': 3,
    '16-20 hours': 4,
    '21 hours above': 5,
    'Seldom study beside the Final Exam Period': 0
}

def prepare_survey(df):
    out = df.copy()
    out['data_source'] = 'survey'

    # Numeric targets
    out['gpa_numeric'] = out[GPA_COL].map(gpa_map)
    out['study_ordinal'] = out[STUDY_COL].map(study_map)

    # Binary pass (GPA >= 3.0)
    out['pass'] = (out['gpa_numeric'] >= 3.0).astype(int)

    # Study level categories
    out['study_level'] = out[STUDY_COL].map({
        'Seldom study beside the Final Exam Period': 'Rare',
        '0-5 hours': 'Low',
        '6-10 hours': 'Low-Med',
        '11-15 hours': 'Medium',
        '16-20 hours': 'High',
        '21 hours above': 'Very High'
    })

    return out

survey_prep = prepare_survey(survey)
print(f'Survey prepared: {survey_prep.shape}')
survey_prep[['gpa_numeric', 'study_ordinal', 'pass', 'study_level']].head()

## Part 3: Prepare Public Set

In [4]:
# Load public dataset
public = pd.read_csv('student_performance_cleaned.csv')
print(f'Public shape: {public.shape}')

def prepare_public(df):
    out = df.copy()
    out['data_source'] = 'public'

    # Convert G3 (0-20) to approximate GPA scale (0-4)
    out['gpa_numeric'] = (out['G3'] / 20) * 4
    out['study_ordinal'] = out['studytime']  # already 1-4
    out['pass'] = (out['G3'] >= 10).astype(int)

    out['study_level'] = out['studytime'].map({
        1: 'Low', 2: 'Medium', 3: 'High', 4: 'Very High'
    })

    # Keep useful columns
    keep = ['sex', 'age', 'studytime', 'failures', 'absences',
            'Medu', 'Fedu', 'internet', 'G3',
            'gpa_numeric', 'study_ordinal', 'pass', 'study_level', 'data_source']
    return out[[c for c in keep if c in out.columns]]

public_prep = prepare_public(public)
print(f'Public prepared: {public_prep.shape}')
public_prep.head(2)

Public shape: (1044, 36)
Public prepared: (1044, 14)


,sex,age,studytime,failures,absences,Medu,Fedu,internet,G3,gpa_numeric,study_ordinal,pass,study_level,data_source
0,F,18,2,0,6,4,4,0,6,1.2,2,0,Medium,public
1,F,17,2,0,4,1,1,1,6,1.2,2,0,Medium,public


## Part 4: Combine & Encode

In [5]:
# Common columns for combining
common = ['gpa_numeric', 'study_ordinal', 'pass', 'study_level', 'data_source']

survey_aligned = survey_prep[common].copy()
public_aligned = public_prep[common].copy()

combined = pd.concat([public_aligned, survey_aligned], ignore_index=True)
print(f'Combined shape: {combined.shape}')
print(combined['data_source'].value_counts())

Combined shape: (1090, 5)
data_source
public    1044
survey      46
Name: count, dtype: int64


In [6]:
# One-hot encode
combined_encoded = pd.get_dummies(
    combined,
    columns=['study_level', 'data_source'],
    drop_first=True
)

print(f'Encoded shape: {combined_encoded.shape}')
combined_encoded.head()

Encoded shape: (1090, 9)


,gpa_numeric,study_ordinal,pass,study_level_Low,study_level_Low-Med,study_level_Medium,study_level_Rare,study_level_Very High,data_source_survey
0,1.2,2,0,False,False,True,False,False,False
1,1.2,2,0,False,False,True,False,False,False
2,2.0,2,1,False,False,True,False,False,False
3,3.0,3,1,False,False,False,False,False,False
4,2.0,2,1,False,False,True,False,False,False


## Part 5: Save Prepared Files

In [7]:
survey_prep.to_csv('features_survey_prepared.csv', index=False)
public_prep.to_csv('features_public_prepared.csv', index=False)
combined.to_csv('features_combined.csv', index=False)
combined_encoded.to_csv('features_combined_encoded.csv', index=False)

print('Saved:')
print('- features_survey_prepared.csv')
print('- features_public_prepared.csv')
print('- features_combined.csv')
print('- features_combined_encoded.csv')

Saved:
- features_survey_prepared.csv
- features_public_prepared.csv
- features_combined.csv
- features_combined_encoded.csv


## Summary

| Dataset | Records | Key Features |
|---------|---------|--------------|
| Survey | 46 | GPA range → numeric, study time → ordinal, pass binary |
| Public | 1,044 | G3 → GPA scale, studytime, failures, absences, etc. |
| Combined | ~1,090 | Aligned features ready for modeling |

**Note:** Survey has only 46 responses. For modeling, the public set is more reliable. Survey data is useful for comparison and validation.